<figure style="text-align:center;">
    <img src="notebook_images/umn_banner.png" alt="University of Minnesota">
</figure>

---
# <center>**<u>CERT x465-003</u>**: Data Interpretation and Application</center><br><center>Coursework and Assignments</center><br><center>Project Title: ***Deep Poking into Pokémon Data***</center>
<figure style="text-align:center;">
    <img src="notebook_images/pokes_pic.jpg" alt="Pokemon" width="1800">
</figure>

---

# **1.)** OVERVIEW/CONTEXT

---
Text goes here....

---

# **2.)** *<u>Software/Libraries Herein Implemented</u>*

---
More text goes here....

---

# **3.)** *<u>Assignments</u>*

---

## **3.1)** <u>Assignment 01</u>
- Completed by: Logan R. Sloan<br>

---
### Assignment 01 Details
- Write a short essay that address the following questions:
    - What problem or question are you trying to address?
    - Why are you interested in addressing this?
    - Why is this an important question or problem to address (in general)?
    - Who might your audience(s) be for this investigation?
    - How will the results of this investigation help your audience make decisions or draw conclusions?<br>

---
### Assignment 01 Essay


## **3.2)** <u>Assignment 02-A</u>
- Completed by: Logan R. Sloan<br>

---
### Assignment 02-A Details
- Identify at least one data source that can help answer the question you developed in Module 1.
1. First, do some research on your own to try and find a data source(s).
    - You might start with a simple Google search, you may ask someone  who has knowledge in the area, or you might already know of some candidates.
    - For example, if your question pertains to K-12 education in Minnesota, you might check out the Department of Education's data sources.
 2. After you have finished your search and identified good data source(s), please write a short essay that addresses to the following questions:
    - Describe the data source(s) that you found.
    - How is the data relevant to your question?
    - Where does the data come from?
    - How reliable is the source? Are there any signs of bias?<br>

---
### Assignment 02-A Essay

# **<u>NOTE</u>**: THE DATA SCRAPING SECTION SHOULD BE MOVED TO ASSIGNMENT 02A, SINCE THE POINT OF THE ASSIGNMENT IS TO FIND A DATASET TO USE AND THAT IS WHAT THE SCRAPING, CLEANING, ET CETERA IS DOING....

## **3.3)** <u>Assignment 03</u>
- Completed by: **Logan R. Sloan**<br>

---
### Assignment 03 Details
- Using the data source(s) that you identified, produce a 5-minute video describing what you are seeing in the data and what conclusions the data is pointing towards.
- Make sure you tie it back to the original question you set out to address.
- In the video, please also include a brief discussion of how groups with different perspectives might interpret the data differently.
- Post this video to share with your peers, and provide feedback on the videos of at least two peers.<br>

---

### **3.3.i)** Data Exploration for Assignment 03

---
#### Overview of Methodology to be Used to Completed Assignment 03
- Since I have chosen to create my own dataset, in order to complete *Assignment 03* the dataset must first be compiled together in a clean, usable format.
    - To do this I will, first, scrape multiple websites (and multiple pages on those site) for the base of my dataset.
    - Then I will cleanup this data where necessary, as scraping always involves data that isn't particularly useful in its "raw" state.
    - Lastly, I will expand upon and abstract from the cleaned data to create more columns/categories within the dataset to better aid my analysis of the dataset.
- To explore the dataset, I will use the paradigm of an Exploratory Data Analysis (EDA) and some other Data Science techniques to dig down into the data and see what interesting points in the pokemon dataset we can poke out and prop up for others' viewing.
    - This will include some comparison based observations, basic and more advanced statics calculations, visualizations, as well as some machine learning (clustering) analysis.

---

### **3.3.ii)** Scraping, Cleaning, and Expanding upon the Data to Build Dataset for Analysis

---

### **3.3.ii.0)** Python GPU Acceleration Activation

---

In [1]:
%load_ext cudf.pandas
%load_ext cuml.accel

 ### **3.3.ii.1)** Python Library Imports for Scraping, Cleaning, and Expanding

---

In [2]:
import os
import io
import re
import time
import urllib.request
import urllib.error
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = 'https://bulbapedia.bulbagarden.net'
DATA_DIR = os.path.expanduser('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/')
_HEADERS = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:132.0) Gecko/20100101 Firefox/132.0'}

 ### **3.3.ii.2)** Base URL to Scrape

In [3]:
 pokemon_ALL_url = "https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_base_stats_in_Generation_IX"

### **3.3.ii.3)** Initial Scraper (Part 1)
- For part 1 of the initial scraper, most of the information from the tables will be pulled from the base url page (see image below for example).
<br>

<figure style="text-align:center;">
    <img src="notebook_images/initial_scrape_img00.png" alt="Bulbapedia Initial Scrape">
    <figcaption>
        The scraper looks through the HTML (right of image) that laysout the webpage.<br>
        The blue highlight shows that this particular part of the table contains both the Pokémon name and the link to it's individual page (to be used by the next scraper).
    </figcaption>
</figure>

In [4]:
def scrape_pokemon_base_stats(url):
    """
    Scrapes the Bulbapedia base stats table using direct HTTP (no headless browser needed
    since the table is server-rendered HTML).
    Table columns: #, Pokémon (sprite), Pokémon.1 (name), HP, Attack, Defense,
                   Sp. Atk, Sp. Def, Speed, Total, Average
    Returns a list of dicts matching the first 10 columns of pokemon_dataset_MASTER.csv.
    """
    req = urllib.request.Request(url, headers={
        'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:132.0) Gecko/20100101 Firefox/132.0'
    })
    with urllib.request.urlopen(req, timeout=30) as resp:
        html = resp.read().decode('utf-8')

    dfs = pd.read_html(io.StringIO(html))
    df = next(
        t for t in dfs
        if 'HP' in t.columns and 'Attack' in t.columns
    )

    records = []
    for _, row in df.iterrows():
        try:
            pokedex_num = int(row['#'])
            name        = str(row['Pokémon.1']).strip()   # .1 suffix = name col; plain 'Pokémon' is the sprite (NaN)
            hp          = int(row['HP'])
            attack      = int(row['Attack'])
            defense     = int(row['Defense'])
            sp_atk      = int(row['Sp. Atk'])
            sp_def      = int(row['Sp. Def'])
            speed       = int(row['Speed'])
            total       = int(row['Total'])
            link = f"{BASE_URL}/wiki/{name.replace(' ', '_')}"
        except (ValueError, TypeError):
            continue

        records.append({
            'Pokedex Number': pokedex_num,
            'Pokemon':        name,
            'Link':           link,
            'HP':             hp,
            'Attack':         attack,
            'Defense':        defense,
            'Speed':          speed,
            'Special Attack': sp_atk,
            'Special Defense': sp_def,
            'Stat Total':     total,
        })

    return records


records = scrape_pokemon_base_stats(pokemon_ALL_url)
pokes_scraped = pd.DataFrame(records)
print(f"Scraped {len(pokes_scraped)} rows.")
pokes_scraped.head(10)

Scraped 1239 rows.


,Pokedex Number,Pokemon,Link,HP,Attack,Defense,Speed,Special Attack,Special Defense,Stat Total
0,1,Bulbasaur,https://bulbapedia.bulbagarden.net/wiki/Bulbasaur,45,49,49,45,65,65,318
1,2,Ivysaur,https://bulbapedia.bulbagarden.net/wiki/Ivysaur,60,62,63,60,80,80,405
2,3,Venusaur,https://bulbapedia.bulbagarden.net/wiki/Venusaur,80,82,83,80,100,100,525
3,3,Venusaur Mega Venusaur,https://bulbapedia.bulbagarden.net/wiki/Venusa...,80,100,123,80,122,120,625
4,4,Charmander,https://bulbapedia.bulbagarden.net/wiki/Charma...,39,52,43,65,60,50,309
5,5,Charmeleon,https://bulbapedia.bulbagarden.net/wiki/Charme...,58,64,58,80,80,65,405
6,6,Charizard,https://bulbapedia.bulbagarden.net/wiki/Charizard,78,84,78,100,109,85,534
7,6,Charizard Mega Charizard X,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,130,111,100,130,85,634
8,6,Charizard Mega Charizard Y,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,104,78,100,159,115,634
9,7,Squirtle,https://bulbapedia.bulbagarden.net/wiki/Squirtle,44,48,65,43,50,64,314


### **3.3.ii.3.x)** Save Initial Dataset

In [5]:
pokes_scraped.to_csv(DATA_DIR + 'pokemon_base_stats_scraped.csv', index=False)
print(f"Saved {len(pokes_scraped)} rows to {DATA_DIR}pokemon_base_stats_scraped.csv")

Saved 1239 rows to /home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_base_stats_scraped.csv


### **3.3.ii.3.xx)** Cleanup Pokémon Names Where "Variations" Exist
- If you look at the preview of the initial scraped dataset, you'll see that some of the Pokémon have variations such as "Mega".
- It's a little off topic and overly technical to show how I identified all these (although this section here could probably be defined the same way).
- But the first code block below searches through the names and where variations exist it assigns the variation name to a new column in the dataset named "Variation".
- The second code block then removes the variation excess from the Pokémon's name, so that the standard Pokémon and the variants have the same name (think how you would categorize black dog and yellow dog).

In [6]:
_VAR_PATTERN = re.compile(
    r'Mega.*[A-Za-z]|Alolan.*[A-Za-z]|Partner.*[A-Za-z]|Galarian.*[A-Za-z]'
    r'|Hisuian.*[A-Za-z]|Paldean.*[A-Za-z]+\D|in L.*[A-Za-z]|Primal.*[A-Za-z]'
    r'|[A-Za-z]+.Forme|[A-Za-z]+.Cloak|[A-Za-z]+.Rotom|[A-Za-z]+.Mode'
    r'|[A-Za-z]+.Kyurem|Ash.*[A-Za-z]|Eternal.*[A-Za-z]|[A-Za-z]+.Size'
    r'|[A-Za-z0-9]+%.Forme|Hoopa(?![\s\S]*?Hoopa).*[A-Za-z]'
    r'|D.*[A-Za-z]+.*Necrozma|[A-Za-z]+.Necrozma|Hero.*[A-Za-z]'
    r'|Crowned.*[A-Za-z]|Eternamax.*[A-Za-z]|\b\w+\s+Rider.*[A-Za-z]'
    r'|Bloodmoon.*[A-Za-z]|Male|Female'
)

variations = []
for name in pokes_scraped['Pokemon']:
    match = _VAR_PATTERN.search(name)
    variations.append(match.group() if match else '')

pokes_scraped.insert(10, 'Variation', variations)

n_var = sum(1 for v in variations if v != '')
print(f"Variation column added. {n_var} variants found out of {len(pokes_scraped)} rows.")
pokes_scraped[pokes_scraped['Variation'] != ''].head(10)

Variation column added. 213 variants found out of 1239 rows.


,Pokedex Number,Pokemon,Link,HP,Attack,Defense,Speed,Special Attack,Special Defense,Stat Total,Variation
3,3,Venusaur Mega Venusaur,https://bulbapedia.bulbagarden.net/wiki/Venusa...,80,100,123,80,122,120,625,Mega Venusaur
7,6,Charizard Mega Charizard X,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,130,111,100,130,85,634,Mega Charizard X
8,6,Charizard Mega Charizard Y,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,104,78,100,159,115,634,Mega Charizard Y
12,9,Blastoise Mega Blastoise,https://bulbapedia.bulbagarden.net/wiki/Blasto...,79,103,120,78,135,115,630,Mega Blastoise
19,15,Beedrill Mega Beedrill,https://bulbapedia.bulbagarden.net/wiki/Beedri...,65,150,40,145,15,80,495,Mega Beedrill
23,18,Pidgeot Mega Pidgeot,https://bulbapedia.bulbagarden.net/wiki/Pidgeo...,83,80,80,121,135,80,579,Mega Pidgeot
25,19,Rattata Alolan Form,https://bulbapedia.bulbagarden.net/wiki/Rattat...,30,56,35,72,25,35,253,Alolan Form
27,20,Raticate Alolan Form,https://bulbapedia.bulbagarden.net/wiki/Ratica...,75,71,70,77,40,80,413,Alolan Form
33,25,Pikachu Partner Pikachu,https://bulbapedia.bulbagarden.net/wiki/Pikach...,45,80,50,120,75,60,430,Partner Pikachu
35,26,Raichu Alolan Form,https://bulbapedia.bulbagarden.net/wiki/Raichu...,60,85,50,110,95,85,485,Alolan Form


In [7]:
pattern = r'Mega.*[A-Za-z]|Alolan.*[A-Za-z]|Partner.*[A-Za-z]|Galarian.*[A-Za-z]|Hisuian.*[A-Za-z]|Paldean.*[A-Za-z]+\D|in L.*[A-Za-z]|Primal.*[A-Za-z]|[A-Za-z]+.Forme|[A-Za-z]+.Cloak|[A-Za-z]+.Rotom|[A-Za-z]+.Mode|[A-Za-z]+.Kyurem|Ash.*[A-Za-z]|Eternal.*[A-Za-z]|[A-Za-z]+.Size|[A-Za-z0-9]+%.Forme|Hoopa(?![\s\S]*?Hoopa).*[A-Za-z]|D.*[A-Za-z]+.*Necrozma|[A-Za-z]+.Necrozma|Hero.*[A-Za-z]|Crowned.*[A-Za-z]|Eternamax.*[A-Za-z]|\b\w+\s+Rider.*[A-Za-z]|Bloodmoon.*[A-Za-z]|Male|Female'

# Removes the matching parts and clean up whitespace
pokes_scraped['Pokemon'] = pokes_scraped['Pokemon'].str.replace(pattern, '', regex=True).str.strip()

<figure style="text-align:center;">
    <img src="notebook_images/great-success.gif" alt="Great Success! Very Nice!">
    <figcaption>
        Very Nice! Part 1 of the initial scraping is completed!<br>
        Here's a trophy: <span style="display: inline-block; transform: rotate(180deg);">&#x1F3C6;</span>
    </figcaption>
</figure>

---

### **3.3.ii.4)** Initial Scraper (Part 2)
- For part 2 of the initial scraper, the links gathered for each individual Pokémon will all have to be gone through to pull more basic information to expand the dataset.
- Where part one was scraping one webpage, in part 2, one thousand two-hundred thirty-nine (1239 &#x1F480;) webpages will need to be scraped.
- The information to be gathered is Category, Type 1, Type 2, Height (m), and Weight (kg).
- Below is an example from one of these webpages.<br><br>

<figure style="text-align:center;">
    <img src="notebook_images/initial_scrape_img01.png" alt="Second of the Scrapes, Initially....">
    <figcaption>
        <span style="display: inline-block; transform: scale(-1, 1);">&#x1F446;</span>...."Seed Pokémon" under the name is the Category for reference....&#x1F446;
    </figcaption>
</figure>

### **NUMBERS)** Create Column Denoting Pokémon Generation
- There are nine generations of Pokémon apparently, and it might be worthwhile to examine changes over the generations
- The "Pokedex Number" column is the best way to do this because variants have the same number as their standard siblings (as opposed to index values or something else).
- The first line of code creates an empty column named "Generation", the all the following lines fill-in which generation number.

In [ ]:
pokes_scraped['Generation'] = np.nan
pokes_scraped.loc[pokes_scraped['Pokedex Number'] <= 151, 'Generation'] = 1
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 151, 'Generation'] = 2
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 251, 'Generation'] = 3
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 386, 'Generation'] = 4
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 493, 'Generation'] = 5
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 649, 'Generation'] = 6
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 721, 'Generation'] = 7
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 809, 'Generation'] = 8
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 905, 'Generation'] = 9

---

### **NUMBERS)** Pokémon Evolution Stage Scraper
- Pokémon evolve as they get stronger or something: People train them and they grow into more ridiculous versions of themselves and this is termed "evolution" in the lore, I think.
- Tracking how the Pokémon change as they evolve will add great insight into the analysis.
- Pokémon, as far as I can tell, either have a two-stage evolution (first stage changes into second/final) or have a three-stage evolution (where the first stage changes into second/intermediate and second into third/final).
- Also, it should be noted that not all Pokémon evolve; as you will see in the analysis, only about 1/3 of Pokémon evolve into other Pokémon.

### **NUMBERS)** Evolution URLs
- I found separate pages on the same website that organize the two-stage evolutions from the three-stage
- This makes the scraping much easier, but it does require rolling through two webpages instead of a single page, but this written to be extremely fast, so no problem....

In [ ]:
evo3_url = 'https://pokemon.fandom.com/wiki/List_of_Pok%C3%A9mon_by_three-stage_evolution'
evo2_url = 'https://pokemon.fandom.com/wiki/List_of_Pok%C3%A9mon_by_two-stage_evolution'

### **NUMBERS)** Evolution Stage Scraper

In [ ]:
DATA_DIR = os.path.expanduser('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/')


async def get_pokes_rows_pw(page, url):
    """Navigate to url and return all non-header rows from .prettytable tables."""
    await page.goto(url, wait_until='domcontentloaded')
    await page.wait_for_selector('.mw-parser-output .prettytable')
    rows = []
    for table in await page.locator('.mw-parser-output .prettytable').all():
        for i, row in enumerate(await table.locator('tr').all()):
            if i > 0:
                rows.append(row)
    return rows


async def get_name(cell):
    """Return the Pokémon name from a table cell (second anchor if present, else first, else cell text)."""
    anchors = await cell.locator('a').all()
    if len(anchors) >= 2:
        return await anchors[1].inner_text()
    elif len(anchors) == 1:
        return await anchors[0].inner_text()
    return await cell.inner_text()


async def pokes_parser1_pw(row):
    """Parse a three-stage evolution table row → dict with ev1/ev2/ev3 keys."""
    cells = await row.locator('td').all()
    n = len(cells)
    if n == 3:
        return {'ev1': await get_name(cells[0]), 'ev2': await get_name(cells[1]), 'ev3': await get_name(cells[2])}
    elif n == 2:
        return {'ev2': await get_name(cells[0]), 'ev3': await get_name(cells[1])}
    else:
        return {'ev3': await get_name(cells[0])}


async def pokes_parser2_pw(row):
    """Parse a two-stage evolution table row → dict with ev1/ev2 keys."""
    cells = await row.locator('td').all()
    n = len(cells)
    if n == 2:
        first_anchors = await cells[0].locator('a').all()
        ev1 = await first_anchors[1].inner_text() if len(first_anchors) >= 2 else await cells[0].inner_text()
        ev2 = await get_name(cells[1])
        return {'ev1': ev1, 'ev2': ev2}
    else:
        cell_anchors = await cells[0].locator('a').all()
        ev2 = await cell_anchors[1].inner_text() if len(cell_anchors) >= 2 else await cells[0].inner_text()
        return {'ev2': ev2}


async def main():
    async with async_playwright() as pw:
        browser = await pw.firefox.launch(headless=True)
        page = await browser.new_page()

        rows3 = await get_pokes_rows_pw(page, evo3_url)
        evo3_data = [await pokes_parser1_pw(r) for r in rows3]
        pd.DataFrame(evo3_data).to_csv(DATA_DIR + 'evo3.csv', index=False)

        rows2 = await get_pokes_rows_pw(page, evo2_url)
        evo2_data = [await pokes_parser2_pw(r) for r in rows2]
        pd.DataFrame(evo2_data).to_csv(DATA_DIR + 'evo2.csv', index=False)

        await browser.close()

await main()
print("Scraping complete. CSVs saved to data directory.")

### **NUMBERS)** Cleaning-up the Evolution Data {***<u>WARNING</u>***: Long-winded explanation of what's happening below--PLEASE SKIP!!!}
- The scraper above creates two seperate `.csv` files (one for each url).
- So, the two will need to be merged together and then merged into the larger, main dataset, but before that can be done, the evolution stage data needs to be turned into integers.
- The reason why the data wasn't scraped as integers is because Python doesn't allow naming variables with only numbers.
- Python variables must start with either an uppercase or lowercase letter [a-z, A-Z] or an underscore [ _ ].
- And when pulling information from a website like this, the data must be stored as a variable to be compiled together.
- First, the data from each website is compiled into what's called a dictionary.
    - Dictionaries work sort of like cabinets:
        1. There are *unique* "keys" that function like each cabinet box in your kitchen [you cannot have keys of the same value just like every cabinet is a totally separate space], and
        2. There are *non-unique* "values" within those keys that can repeat (like having two of the same pots and whatnot).
- Second, the two separate dictionaries are saved as `.csv` files.
    - A `.csv` is used for three reasons:
        1.  In case something messes up in the integer conversion or merging, we won't need to run the scraper again (and can simply work on fixing our crap code).
        2. because `.csv` files can easily loaded into what are called DataFrames, and these are very easy to manipulate and merge together.
        3. There are certain Pokémon that have the ability to evolve into more than one, singular Pokémon in the next stage, and the way the website's table is setup, I could not find a good work around to match these separate evolutions.
            - [Reference the parenthetical from #1]--This being the case and my code being crap, a `.csv` file is human-readable and easy to manually edit.
            - If you *must* know, it was easy to edit because the Pokémon missing their evolution stages are directly below the stages they are missing, so I just ran a script that found blanks, moved a row up, copied the correct number of values, and pasted them in the blanks.
            - See the picture below to better understand how the table is set up: because of how HTML functions, the Pokémon "Bellossom" is technically on a row by itself, where as "Vileplume" is rowed with the first two.
<figure style="text-align:center;">
    <img src="notebook_images/evo_scrape_img00.png" alt="Gloom of HTML">
    <figcaption>"Gloom" is my natural habitat....
        <span style="display: inline-block; transform: scale(1.5, -1);">&#x1F984;</span>
    </figcaption>
</figure>
<br>

---
- This all means that cleaning the code will work in this order:
    1. Manually filling in the missing Pokémon for the edge cases.
    2. Loading in the separate `.csv` files.
    3. Merging, one-at-a-time, the two `.csv` files into the main dataset
    4. Creating two new columns in the main dataset from the non-integer data (which works to convert it to integers in the process).
        1. <u>"Evolution Stage"</u>: This is the integer value of the Pokémon's place in the evolution chain (1, 2, or 3--and 0 for Pokémon that don't evolve).
        2. <u>"Evolves"</u>: This is a boolean value (i.e. True or False) that tells us whether a Pokémon evolves or not. This is to make it easier to analyze non-evolving Pokémon vs. evolving Pokémon.

In [ ]:
evo3 = pd.read_csv(DATA_DIR + 'evo3.csv')
evo2 = pd.read_csv(DATA_DIR + 'evo2.csv')


# Merge three-stage evolution data into df
df['ev1'] = df['Pokemon'].isin(evo3['ev1'])
df['ev2'] = df['Pokemon'].isin(evo3['ev2'])
df['ev3'] = df['Pokemon'].isin(evo3['ev3'])

df.loc[df['ev1'] == True, ['Evolution Stage', 'Evolves']] = [1, True]
df.loc[df['ev2'] == True, ['Evolution Stage', 'Evolves']] = [2, True]
df.loc[df['ev3'] == True, ['Evolution Stage', 'Evolves']] = [3, False]

df.drop(['ev1', 'ev2', 'ev3'], axis=1, inplace=True)

# Merge two-stage evolution data into df
df['ev1'] = df['Pokemon'].isin(evo2['ev1'])
df['ev2'] = df['Pokemon'].isin(evo2['ev2'])

df.loc[df['ev1'] == True, ['Evolution Stage', 'Evolves']] = [1, True]
df.loc[df['ev2'] == True, ['Evolution Stage', 'Evolves']] = [2, False]

df.drop(['ev1', 'ev2'], axis=1, inplace=True)

# Fill remaining NaN evolution stages (non-evolving Pokémon)
df.loc[df['Evolution Stage'].isna(), ['Evolution Stage', 'Evolves']] = [0, False]

print("Cleaning and merging complete.")

### **NUMBERS)** Probably a Good Time to Save Our Progress

In [ ]:
pokes_scraped.to_csv(('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_dataset_MASTER.csv', index=False))

###

### **3.3.iii)** Exploratory Data Analysis (EDA)

---

#### EDA Overview
- This section contains the full Exploratory Data Analysis for Assignment 03.
- The dataset used is `pokemon_dataset_MASTER.csv` (1,231 rows × 55 columns), custom-built from scraped Bulbapedia and Pokémon Fandom Wiki data.
- Eight thematic questions are explored, followed by an unsupervised machine learning clustering capstone.

#### GPU ACCELERATION

In [2]:
%load_ext cudf.pandas
%load_ext cuml.accel

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


In [3]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde as _gaussian_kde
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

STAT_COLS   = ['HP', 'Attack', 'Defense', 'Speed', 'Special Attack', 'Special Defense']
ROLE_ORDER  = ['Physical Sweeper', 'Special Sweeper', 'Mixed Attacker',
               'Balanced', 'Tank', 'Physical Wall', 'Special Wall']
PCT_COLS    = ['HP_pct', 'Attack_pct', 'Defense_pct',
               'Speed_pct', 'Special_Attack_pct', 'Special_Defense_pct']
ZSCORE_COLS = ['HP_zscore', 'Attack_zscore', 'Defense_zscore',
               'Speed_zscore', 'Special_Attack_zscore', 'Special_Defense_zscore']

DATA_DIR = os.path.expanduser(
    '~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/'
)
pokes = pd.read_csv(DATA_DIR + 'pokemon_dataset_MASTER.csv')

# Chart defaults applied to every figure
_H = 800
_W = 1800
_T = 'plotly_dark'

In [4]:
# --- Official Pokémon Type Color Palette ---
TYPE_COLORS = {
    'Normal': '#A8A77A',   'Fire': '#EE8130',    'Water': '#6390F0',
    'Electric': '#F7D02C', 'Grass': '#7AC74C',   'Ice': '#96D9D6',
    'Fighting': '#C22E28', 'Poison': '#A33EA1',  'Ground': '#E2BF65',
    'Flying': '#A98FF3',   'Psychic': '#F95587', 'Bug': '#A6B91A',
    'Rock': '#B6A136',     'Ghost': '#735797',   'Dragon': '#6F35FC',
    'Dark': '#705746',     'Steel': '#B7B7CE',   'Fairy': '#D685AD',
}

def _hex_scale(hex_color, factor):
    """factor < 1 = darken, factor > 1 = lighten toward white."""
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    if factor > 1:
        f = factor - 1
        r = int(r + (255 - r) * f)
        g = int(g + (255 - g) * f)
        b = int(b + (255 - b) * f)
    else:
        r, g, b = int(r * factor), int(g * factor), int(b * factor)
    return '#{:02x}{:02x}{:02x}'.format(min(r, 255), min(g, 255), min(b, 255))

TYPE_COLORS_LIGHT = {t: _hex_scale(c, 1.45) for t, c in TYPE_COLORS.items()}
TYPE_COLORS_DARK  = {t: _hex_scale(c, 0.60) for t, c in TYPE_COLORS.items()}

ROLE_COLORS   = ['#66C2A5', '#FC8D62', '#8DA0CB', '#E78AC3', '#A6D854', '#FFD92F', '#E5C494']
ROLE_COLOR_MAP = dict(zip(ROLE_ORDER, ROLE_COLORS))

def stats_note(fig, text, y_pos=-0.1, bottom_margin=69):
    """Append a centered stats block below the chart area."""
    fig.update_layout(margin=dict(b=bottom_margin))
    fig.add_annotation(
        text=text, xref='paper', yref='paper',
        x=0.5, y=y_pos, showarrow=False,
        font=dict(size=13, color='#cccccc'), align='center',
        bgcolor='rgba(30,30,30,0.75)',
        bordercolor='rgba(200,200,200,0.35)', borderwidth=1, borderpad=7,
    )
    return fig

print(f"TYPE_COLORS loaded: {len(TYPE_COLORS)} types")

TYPE_COLORS loaded: 18 types


In [4]:
# --- Dataset Verification ---
assert pokes.shape == (1231, 55), f"Expected (1231, 55), got {pokes.shape}"
assert 'BMI' in pokes.columns,             "Missing column: BMI"
assert 'Role' in pokes.columns,            "Missing column: Role"
assert 'Stat Std Dev' in pokes.columns,    "Missing column: Stat Std Dev"
assert 'Legendary' in pokes.columns,       "Missing column: Legendary"
assert 'Evolution Stage' in pokes.columns, "Missing column: Evolution Stage"
for col in ZSCORE_COLS:
    assert col in pokes.columns, f"Missing z-score column: {col}"

print(f"Dataset loaded: {pokes.shape[0]} rows × {pokes.shape[1]} columns")
print(f"\nRole distribution:\n{pokes['Role'].value_counts().to_string()}")
print(f"\nMissing values in key columns:\n"
      f"{pokes[['BMI','Role','Stat Std Dev','Legendary','Evolution Stage']].isna().sum().to_string()}")

Dataset loaded: 1231 rows × 55 columns

Role distribution:
Role
Tank                389
Balanced            355
Special Sweeper     124
Mixed Attacker      100
Physical Sweeper     97
Physical Wall        85
Special Wall         81
Name: count, dtype: int64

Missing values in key columns:
BMI                0
Role               0
Stat Std Dev       0
Legendary          0
Evolution Stage    0
dtype: int64


### **3.3.iii.1)** Section 1: Body Composition and Combat Role

---
**Question:** Does a Pokémon's physical body shape (BMI, height, weight) predict how it fights?

In [40]:
# --- Combat Role Definitions ---
role_defs = {
    'Physical Sweeper':  'High Attack and Speed — hits first and hard with physical moves',
    'Special Sweeper':   'High Special Attack and Speed — same offensive intent through the special damage channel',
    'Mixed Attacker':    'High in both Attack and Special Attack — threats that cannot be walled by a single defensive stat',
    'Balanced':          'No dominant stat — generalist profile without a clear offensive or defensive identity',
    'Tank':              'High HP and both defensive stats — absorbs hits from any angle while retaining offensive presence',
    'Physical Wall':     'Specialized in Defense and HP — shuts down physical attackers at the cost of special vulnerability',
    'Special Wall':      'Specialized in Special Defense and HP — the mirror of Physical Wall',
}

header_color  = '#735797'
row_colors    = ['#B7B7CE', '#9696a8']

fig = go.Figure(go.Table(
    columnwidth=[160, 700],
    header=dict(
        values=['<b>Role</b>', '<b>Definition</b>'],
        fill_color=header_color,
        font=dict(color='white', size=33),
        align='center',
        height=36,
    ),
    cells=dict(
        values=[
            list(role_defs.keys()),
            list(role_defs.values()),
        ],
        fill_color=[[row_colors[i % 2] for i in range(len(role_defs))]],
        font=dict(color='white', size=23),
        align='left',
        height=32,
    ),
))
fig.update_layout(
    title='',
    title_x=0.5,
    template=_T,
    height=525,
    width=1300,
    margin=dict(l=20, r=20, t=60, b=20),
)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.9)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.9)'),
)

fig.write_image('notebook_images/plot_images/role_table.png', scale=2)
fig.show()

In [66]:
# --- 1a: BMI Distribution ---
bmi_all  = pokes['BMI'].clip(upper=200)
bmi_filt = pokes.loc[pokes['BMI'] < 200, 'BMI']

fig = make_subplots(rows=1, cols=2,
                    horizontal_spacing=0.19,
                    subplot_titles=['BMI Distribution (clipped at 200)',
                                    'BMI Distribution (BMI < 200)'])

for col_idx, (vals, clr) in enumerate(
        [(bmi_all, '#64AD9A'), (bmi_filt, '#724696')], 1):
    fig.add_trace(go.Histogram(
        x=vals, nbinsx=60,
        marker_color=clr, opacity=0.9,
        marker_line_width=0.77, marker_line_color='rgba(0,0,0,0.4)',
        showlegend=False,
    ), row=1, col=col_idx)
    kde = _gaussian_kde(vals.dropna())
    x_kde = np.linspace(float(vals.min()), float(vals.max()), 300)
    y_kde = kde(x_kde)
    bin_w = (float(vals.max()) - float(vals.min())) / 60
    fig.add_trace(go.Scatter(
        x=x_kde, y=y_kde * len(vals) * bin_w,
        mode='lines', line=dict(color='white', width=2),
        showlegend=False,
    ), row=1, col=col_idx)

fig.update_layout(
    title=dict(text='<b>Pokémon BMI Distribution</b>', x=0.5, font=dict(size=20)),
    template=_T, height=600, width=1500, bargap=0,
)

top10 = pokes.nlargest(10, 'BMI')[['Pokemon', 'Type 1', 'Height (m)', 'Weight (kg)', 'BMI', 'Role']]
note = '<b>Top 10 Highest-BMI Pokémon:</b><br>' + '<br>'.join(
    f"{r.Pokemon} ({r['Type 1']})  BMI={r.BMI:.1f}<br>Role: {r.Role}"
    for _, r in top10.iterrows()
)
stats_note(fig, note, y_pos=0.5, bottom_margin=50)
fig.write_image('notebook_images/plot_images/BMI_dist.png', scale=2)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.9)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.9)'),
    xaxis2=dict(gridcolor='rgba(0,0,0,0.9)'),
    yaxis2=dict(gridcolor='rgba(0,0,0,0.9)'),
)

fig.write_image('notebook_images/plot_images/BMI_dist.png', scale=2)
fig.show()

In [4]:
# --- 1b: BMI by Combat Role ---
bmi_data = pokes[pokes['BMI'] < 200].copy()

fig = go.Figure()
for role in ROLE_ORDER:
    subset = bmi_data[bmi_data['Role'] == role]['BMI']
    fig.add_trace(go.Violin(
        y=subset, name=role,
        fillcolor=ROLE_COLOR_MAP.get(role, '#888888'),
        line=dict(color='black', width=3),
        opacity=0.75, box_visible=True,
        meanline_visible=True, points=False,
    ))

fig.update_layout(
    title=dict(text='<b>BMI Distribution by Combat Role</b>', x=0.5, font=dict(size=20)),
    xaxis_title='Combat Role', yaxis_title='BMI',
    template=_T, height=676, width=1800,
    violinmode='overlay', showlegend=False,
)

role_stats = bmi_data.groupby('Role')['BMI'].agg(['median', 'mean']).reindex(ROLE_ORDER)
note = '  |  '.join(
    f"{r}: median={row['median']:.1f}" for r, row in role_stats.iterrows()
)
stats_note(fig, f'<b>Median BMI by Role:</b><br>{note}', y_pos=-.23, bottom_margin=123)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.9)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.9)'),
)

fig.write_image('notebook_images/plot_images/BMI_dist-vs-role.png', scale=2)
fig.show()


In [53]:
# --- 1c: Mean Height vs. Mean Weight by Primary Type ---
type_hw = (
    pokes.groupby('Type 1')
    .agg(
        Height=('Height (m)', 'mean'),
        Weight=('Weight (kg)', 'mean'),
        BMI=('BMI', 'median'),
        Count=('Pokemon', 'count'),
    )
    .reset_index()
)

fig = px.scatter(
    type_hw,
    x='Height', y='Weight',
    size='Count',
    color='Type 1',
    color_discrete_map=TYPE_COLORS,
    text='Type 1',
    hover_name='Type 1',
    hover_data={'Height': ':.2f', 'Weight': ':.1f', 'BMI': ':.1f', 'Count': True, 'Type 1': False},
    log_x=True, log_y=True,
    title='<b>Mean Height vs. Mean Weight by Primary Type (log-log scale)</b><br>'
          '<sup>Bubble size = number of Pokémon of that type</sup>',
    template=_T,
    height=_H, width=1300,
)
fig.update_traces(textposition='top center', textfont=dict(size=11))
fig.update_layout(title_x=0.5, showlegend=False)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.9)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.9)'),
)

fig.write_image('notebook_images/plot_images/height-v-weight_bytype.png', scale=2)
fig.show()

In [62]:
# --- 1d: Median & Mean BMI by Generation ---
# Cosmoem (BMI=99,990) is excluded from mean — a 0.1m / 999.9kg outlier that
# inflates Gen 7's mean from 40.8 to 1,030.4 single-handedly.
pokes_bmi = pokes[pokes['Pokemon'] != 'Cosmoem']

gen_bmi = (
    pokes_bmi.groupby('Generation')['BMI']
    .agg(Median='median', Mean='mean')
    .reset_index()
    .melt(id_vars='Generation', var_name='Metric', value_name='BMI')
)

fig = px.bar(
    gen_bmi, x='Generation', y='BMI',
    color='Metric',
    color_discrete_map={'Median': '#64AD9A', 'Mean': '#C296E6'},
    barmode='group',
    title='<b>Pokémon BMI by Generation</b>',
    labels={'BMI': 'BMI'},
    template=_T, height=1000, width=1400,
)
fig.update_layout(
   title=dict( x=0.5, font=dict(size=20)),
    xaxis=dict(tickmode='linear', tick0=1, dtick=1),
    legend=dict(title='Metric'),
)

gen_bmi_wide = gen_bmi.pivot(index='Generation', columns='Metric', values='BMI')
sep = '  |  '
row_gen    = sep.join(f"Gen {int(g)}"           for g        in gen_bmi_wide.index)
row_median = sep.join(f"{row.Median:.1f}"        for _, row   in gen_bmi_wide.iterrows())
row_mean   = sep.join(f"{row.Mean:.1f}"          for _, row   in gen_bmi_wide.iterrows())
note = f"<b>BMI per Generation</b> (Cosmoem excluded — BMI 99,990 inflates Gen 7 mean from 40.8 → 1,030)<br>{row_gen}<br>median: {row_median}<br>mean:   {row_mean}"
stats_note(fig, note, y_pos=-.23, bottom_margin=222)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.9)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.9)'),
)

fig.write_image('notebook_images/plot_images/BMI_generation.png', scale=2)
fig.show()

### **3.3.iii.2)** Section 2: Type Shapes Stat Identity, Not Just Stat Total

---
**Question:** Does a Pokémon's primary type determine its fighting personality more than its raw power level?

In [80]:
# --- 2a: Combat ratio profiles by Type 1 ---
type_ps = pokes.groupby('Type 1')['Physical/Special'].median().sort_values()
type_od = pokes.groupby('Type 1')['Offensive/Defensive'].median().sort_values()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Median Physical/Special Ratio by Type',
                                    'Median Offensive/Defensive Ratio by Type'])

fig.add_trace(go.Bar(
    y=type_ps.index.tolist(), x=type_ps.values, orientation='h',
    marker_color=[TYPE_COLORS.get(t, '#888') for t in type_ps.index],
    showlegend=False,
), row=1, col=1)
fig.add_vline(x=1.0, line_dash='dash', line_color='red', row=1, col=1)

fig.add_trace(go.Bar(
    y=type_od.index.tolist(), x=type_od.values, orientation='h',
    marker_color=[TYPE_COLORS.get(t, '#888') for t in type_od.index],
    showlegend=False,
), row=1, col=2)
fig.add_vline(x=1.0, line_dash='dash', line_color='red', row=1, col=2)

fig.update_layout(
    title=dict(text='<b>Combat Personality by Primary Type</b>', x=0.5, font=dict(size=20)),
    template=_T, height=900, width=1400,
)
fig.update_xaxes(title_text='Ratio  (dashed = balanced at 1.0)')

note = (
    f'Most physical: {type_ps.idxmax()} ({type_ps.max():.2f})  |  '
    f'Most special: {type_ps.idxmin()} ({type_ps.min():.2f})<br>'
    f'Most offensive: {type_od.idxmax()} ({type_od.max():.2f})  |  '
    f'Most defensive: {type_od.idxmin()} ({type_od.min():.2f})'
)
stats_note(fig, note, y_pos=-.13, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    xaxis2=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis2=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/combat-personality_type.png', scale=2)
fig.show()

In [81]:
# --- 2a-ii: Combat ratio profiles by Type 2 ---
pokes_t2 = pokes[pokes['Type 2'].notna()]
type2_ps = pokes_t2.groupby('Type 2')['Physical/Special'].median().sort_values()
type2_od = pokes_t2.groupby('Type 2')['Offensive/Defensive'].median().sort_values()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Median Physical/Special Ratio by Type',
                                    'Median Offensive/Defensive Ratio by Type'])

fig.add_trace(go.Bar(
    y=type2_ps.index.tolist(), x=type2_ps.values, orientation='h',
    marker_color=[TYPE_COLORS.get(t, '#888') for t in type2_ps.index],
    showlegend=False,
), row=1, col=1)
fig.add_vline(x=1.0, line_dash='dash', line_color='red', row=1, col=1)

fig.add_trace(go.Bar(
    y=type2_od.index.tolist(), x=type2_od.values, orientation='h',
    marker_color=[TYPE_COLORS.get(t, '#888') for t in type2_od.index],
    showlegend=False,
), row=1, col=2)
fig.add_vline(x=1.0, line_dash='dash', line_color='red', row=1, col=2)

fig.update_layout(
    title=dict(text='<b>Combat Personality by Secondary Type</b>', x=0.5, font=dict(size=20)),
    template=_T, height=900, width=1400,
)
fig.update_xaxes(title_text='Ratio  (dashed = balanced at 1.0)')

note = (
    f'Most physical: {type2_ps.idxmax()} ({type2_ps.max():.2f})  |  '
    f'Most special: {type2_ps.idxmin()} ({type2_ps.min():.2f})<br>'
    f'Most offensive: {type2_od.idxmax()} ({type2_od.max():.2f})  |  '
    f'Most defensive: {type2_od.idxmin()} ({type2_od.min():.2f})'
)
stats_note(fig, note, y_pos=-.13, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    xaxis2=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis2=dict(gridcolor='rgba(255,255,255,0.69)'),
)

fig.write_image('notebook_images/plot_images/combat-personality_type2.png', scale=2)
fig.show()

In [7]:
# --- 2b: Stat fingerprint radar charts (all types) ---
ALL_TYPES  = list(TYPE_COLORS.keys())
STAT_THETA = ['HP', 'Attack', 'Defense', 'Speed', 'Special Attack', 'Special Defense']

fig = make_subplots(
    rows=3, cols=6,
    specs=[[{'type': 'polar'}] * 6] * 3,
    subplot_titles=[f'<b>{t}</b>' for t in ALL_TYPES],
    vertical_spacing=0.05, horizontal_spacing=0.06,
)

polar_max = int(np.ceil(pokes[STAT_THETA].max().max() / 10) * 10 - 130)

for i, ptype in enumerate(ALL_TYPES):
    row, col = divmod(i, 6)
    row += 1; col += 1
    means = pokes[pokes['Type 1'] == ptype][STAT_THETA].mean().tolist()
    clr   = TYPE_COLORS.get(ptype, '#888888')
    fig.add_trace(go.Scatterpolar(
        r=means + [means[0]],
        theta=STAT_THETA + [STAT_THETA[0]],
        fill='toself', fillcolor=clr, opacity=0.5,
        line=dict(color='black', width=3),
        name=ptype, showlegend=False,
    ), row=row, col=col)
    key = 'polar' if i == 0 else f'polar{i + 1}'
    fig.update_layout(**{key: dict(
        bgcolor='rgba(0,0,0,0)',
        radialaxis=dict(range=[0, polar_max], tickfont=dict(size=8)),
    )})

fig.update_layout(
    title=dict(text='<b>Stat Profile "Fingerprint" by Type</b>', x=0.5, font=dict(size=20)),
    template=_T, height=1000, width=2000,
)
fig.update_annotations(font_size=16)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
)

fig.write_image('notebook_images/plot_images/stat_fingerprint.png', scale=2)
fig.show()

In [12]:
# --- 2c: Stat fingerprint ScatterPolar — Primary vs. Secondary type ---
# Type 2 drawn FIRST (behind): darker shade, opacity=0.72, thick solid white border
# Type 1 drawn SECOND (on top): lighter shade, opacity=0.45, thick solid black border
STAT_THETA = ['HP', 'Attack', 'Defense', 'Speed', 'Special Attack', 'Special Defense']
TYPE_ORDER = list(TYPE_COLORS.keys())

all_means = []
for t in TYPE_ORDER:
    for subset in [pokes[pokes['Type 1'] == t], pokes[pokes['Type 2'] == t]]:
        if len(subset) > 0:
            all_means.extend(subset[STAT_THETA].mean().tolist())
POLAR_MAX = int(np.ceil(max(all_means) / 10) * 10)

ROWS, COLS = 3, 6
fig = make_subplots(
    rows=ROWS, cols=COLS,
    specs=[[{'type': 'polar'}] * COLS] * ROWS,
    subplot_titles=[f'<b>{t}</b>' for t in TYPE_ORDER],
    vertical_spacing=0.03,
    horizontal_spacing=0.0888,
)

for idx, t in enumerate(TYPE_ORDER):
    row = idx // COLS + 1
    col = idx % COLS + 1

    # --- Type 2 FIRST (drawn behind): dark shade, high opacity, thick border (white) ---
    t2 = pokes[pokes['Type 2'] == t]
    if len(t2) > 0:
        fig.add_trace(go.Scatterpolar(
            r=t2[STAT_THETA].mean().tolist(),
            theta=STAT_THETA,
            fill='toself',
            fillcolor=TYPE_COLORS_DARK[t],
            opacity=0.72,
            line=dict(color='white', width=3),
            name='Secondary (Type 2)',
            legendgroup='type2',
            showlegend=(idx == 0),
        ), row, col)

    # --- Type 1 SECOND (drawn on top): light shade, lower opacity, thick border (black) ---
    t1 = pokes[pokes['Type 1'] == t]
    if len(t1) > 0:
        fig.add_trace(go.Scatterpolar(
            r=t1[STAT_THETA].mean().tolist(),
            theta=STAT_THETA,
            fill='toself',
            fillcolor=TYPE_COLORS_LIGHT[t],
            opacity=0.45,
            line=dict(color='black', width=3),
            name='Primary (Type 1)',
            legendgroup='type1',
            showlegend=(idx == 0),
        ), row, col)

polar_kwargs = {}
for i in range(1, 19):
    key = 'polar' if i == 1 else f'polar{i}'
    polar_kwargs[key] = dict(
        bgcolor='rgba(0,0,0,0)',
        radialaxis=dict(
            range=[0, POLAR_MAX], showticklabels=True, tickfont=dict(size=8),
        ),
    )

fig.update_layout(
    **polar_kwargs,
    title=dict(
        text=(
            '<b>Mean Stat Profile by Type: Primary vs. Secondary</b>'
            '<br>'
            '<sup>Dark solid fill = Secondary type (Type 2) drawn first  ·  '
            'Light dotted = Primary type (Type 1) on top</sup>''<br>'
        ),
        x=0.5, y=.96, yanchor='top', font=dict(size=20),
    ),
    height=1100,
    width=2000,
    template=_T,
    margin=dict(t=123),
    font=dict(size=16),
)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
)

fig.write_image('notebook_images/plot_images/stat_fingerprint_type2.png', scale=2)
fig.show()

In [79]:
# --- 2d: Does dual typing affect stat profile shape? ---
pokes['Type Label'] = pokes['Dual Type'].map({True: 'Dual Type', False: 'Mono Type'})

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Physical/Special by Type Count',
                                    'Offensive/Defensive by Type Count'])

for col_idx, metric in enumerate(['Physical/Special', 'Offensive/Defensive'], 1):
    for label, clr in [('Mono Type', '#5B9BD5'), ('Dual Type', '#F28B82')]:
        subset = pokes[pokes['Type Label'] == label][metric]
        fig.add_trace(go.Violin(
            y=subset, name=label,
            fillcolor=clr,
            line=dict(color='white', width=3),
            opacity=0.75, box_visible=True,
            meanline_visible=True, points=False,
            legendgroup=label,
            showlegend=(col_idx == 1),
        ), row=1, col=col_idx)
    fig.add_hline(y=1.0, line_dash='dash', line_color='rgba(255,255,0,0.5)', row=1, col=col_idx)

fig.update_layout(
    title=dict(text='<b>Does Dual Typing Affect Stat Identity?</b>', x=0.5, font=dict(size=20)),
    template=_T, height=666, width=1234,
    violinmode='group',
    violingap=0.05, violingroupgap=0.025,
)

medians = pokes.groupby('Type Label')[['Physical/Special', 'Offensive/Defensive']].median().round(3)
means   = pokes.groupby('Type Label')[['Physical/Special', 'Offensive/Defensive']].mean().round(3)
note = (
    f"<b>Mono Type</b> — "
    f"Phys/Spec: median {medians.loc['Mono Type','Physical/Special']}  mean {means.loc['Mono Type','Physical/Special']}  |  "
    f"Off/Def: median {medians.loc['Mono Type','Offensive/Defensive']}  mean {means.loc['Mono Type','Offensive/Defensive']}<br>"
    f"<b>Dual Type</b> — "
    f"Phys/Spec: median {medians.loc['Dual Type','Physical/Special']}  mean {means.loc['Dual Type','Physical/Special']}  |  "
    f"Off/Def: median {medians.loc['Dual Type','Offensive/Defensive']}  mean {means.loc['Dual Type','Offensive/Defensive']}"
)
stats_note(fig, note, y_pos=-.23, bottom_margin=113)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    xaxis2=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis2=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/stat_profile_types.png', scale=2)
fig.show()

### **3.3.iii.3)** Section 3: The Specialization Premium

---
**Question:** Do more powerful Pokémon pay for their power by becoming one-dimensional?

In [93]:
# --- 3a: Stat Std Dev vs. Stat Total ---
tier_clr_map = {'Low': '#91CF60', 'Mid': '#FEE08B', 'High': '#FC8D59', 'Very High': '#D73027'}

fig = px.scatter(
    pokes.dropna(subset=['Stat Total', 'Stat Std Dev', 'Stat Tier']),
    x='Stat Total', y='Stat Std Dev',
    color='Stat Tier',
    color_discrete_map=tier_clr_map,
    hover_name='Pokemon',
    hover_data=['Role', 'Type 1', 'Legendary'],
    trendline='ols',
    trendline_scope='overall',
    trendline_color_override='red',
    title='<b>Does Higher Power = Greater Specialization?</b>',
    template=_T, height=888, width=1600,
    opacity=0.69,
    category_orders={'Stat Tier': ['Low', 'Mid', 'High', 'Very High']},
)
fig.update_layout(title_x=0.5, yaxis_title='Stat Std Dev  (higher = more specialized)',  title=dict(font=dict(size=20)))
fig.update_traces(marker=dict(size=20))

r = np.corrcoef(pokes['Stat Total'].fillna(0), pokes['Stat Std Dev'].fillna(0))[0, 1]
stats_note(fig, f'<b>Pearson r (Stat Total vs. Stat Std Dev):</b>  {r:.4f}', y_pos=-.13, bottom_margin=111)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
)

fig.write_image('notebook_images/plot_images/stat_specialization.png', scale=2)
fig.show()

In [5]:
# --- 3b: Stat Std Dev by Stat Tier and by Role ---
tier_order   = ['Low', 'Mid', 'High', 'Very High']
tier_clr_map = {'Low': '#91CF60', 'Mid': '#FEE08B', 'High': '#FC8D59', 'Very High': '#D73027'}

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Stat Std Dev by Stat Tier', 'Stat Std Dev by Combat Role'])

for tier in tier_order:
    fig.add_trace(go.Box(
        y=pokes[pokes['Stat Tier'] == tier]['Stat Std Dev'],
        name=tier, marker_color=tier_clr_map[tier],
        boxmean='sd', showlegend=False,
    ), row=1, col=1)

for role in ROLE_ORDER:
    fig.add_trace(go.Violin(
        y=pokes[pokes['Role'] == role]['Stat Std Dev'],
        name=role, fillcolor=ROLE_COLOR_MAP.get(role, '#888'),
        line=dict(color='white', width=3),
        opacity=0.75,
        box_visible=True, meanline_visible=True, points=False,
        showlegend=False,
    ), row=1, col=2)

fig.update_layout(
    title=dict(text='<b>The Specialization Premium</b>', x=0.5, font=dict(size=20)),
    yaxis_title='Stat Std Dev', yaxis2_title='Stat Std Dev',
    template=_T, height=666, width=1800,
)

mean_by_tier = pokes.groupby('Stat Tier')['Stat Std Dev'].mean().reindex(tier_order).round(2)
note = '  |  '.join(f"{t}: {v}" for t, v in mean_by_tier.items())
stats_note(fig, f'<b>Mean Stat Std Dev by Tier:</b><br>{note}',  y_pos=-.27, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    xaxis2=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis2=dict(gridcolor='rgba(255,255,255,0.69)'),
)

fig.write_image('notebook_images/plot_images/stat_specialization_roles.png', scale=2)
fig.show()


### **3.3.iii.4)** Section 4: The Speed-Bulk Tradeoff Across Generations

---
**Question:** Is "fast = fragile" a real design rule in the data, and has it intensified over time?

In [20]:
# --- 4a: Speed vs. Bulk  &  Glass Cannon test (Offense vs. Bulk) ---
_4a_scale = ['#64AD9A',  '#8ACBBA', '#F1EE88', '#ECB196', '#3E5E80', '#1A3A50']

pokes['Bulk']    = pokes['HP'] + pokes['Defense'] + pokes['Special Defense']
pokes['Offense'] = pokes['Attack'] + pokes['Special Attack']

assert pokes['Bulk'].isna().sum() == 0,    "Bulk has unexpected NaNs"
assert pokes['Offense'].isna().sum() == 0, "Offense has unexpected NaNs"


fig1 = px.scatter(
    pokes,
    x='Speed', y='Bulk',
    color='Generation',
    color_continuous_scale=_4a_scale,
    hover_name='Pokemon',
    hover_data=['Role', 'Stat Total', 'Type 1'],
    trendline='ols',
    title='<b>Speed vs. Bulk (HP + Def + SpDef)</b><br>'
          '<sup>Tests "fast = fragile" — does Speed trade against defensive staying power?</sup>',
    template=_T, opacity=0.8,
    height=_H, width=_W,
)
fig1.update_layout(title_x=0.5)
fig1.update_traces(marker=dict(size=17))

r_speed_bulk = pokes[['Speed', 'Bulk']].corr().iloc[0, 1]
stats_note(fig1,
    f'<b>Pearson r (Speed vs. Bulk):</b>  {r_speed_bulk:.4f}  '
    f'{"— weak positive: fast Pokémon are NOT systematically fragile" if r_speed_bulk > 0 else "— negative: fast Pokémon tend to be frailer"}',
    y_pos=-0.23, bottom_margin=111)
fig1.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
)

fig1.write_image('notebook_images/plot_images/speed_bulk.png', scale=2)
fig1.show()

In [21]:
# --- 4b: Speed vs. Bulk  &  Glass Cannon test (Offense vs. Bulk) ---
_4a_scale = ['#64AD9A',  '#8ACBBA', '#F1EE88', '#ECB196', '#3E5E80', '#1A3A50']

fig2 = px.scatter(
    pokes,
    x='Offense', y='Bulk',
    color='Generation',
    color_continuous_scale=_4a_scale,
    hover_name='Pokemon',
    hover_data=['Type 1', 'Role', 'Stat Total', 'Speed'],
    trendline='ols',
    title='<b>Offense vs. Bulk (Attack + SpAtk  vs.  HP + Def + SpDef)</b><br>'
          '<sup>Tests the "glass cannon" trope — does raw offensive power trade against survivability?</sup>',
    template=_T, opacity=0.9,
    height=_H, width=_W,
)
fig2.update_layout(title_x=0.5)
fig2.update_traces(marker=dict(size=17))

r_off_bulk = pokes[['Offense', 'Bulk']].corr().iloc[0, 1]
stats_note(fig2,
    f'<b>Pearson r (Offense vs. Bulk):</b>  {r_off_bulk:.4f}  '
    f'{"— positive: offensive Pokémon are NOT glass cannons in the data" if r_off_bulk > 0 else "— negative: the glass cannon trope holds statistically"}',
    y_pos=-0.23, bottom_margin=111)
fig2.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
)

fig2.write_image('notebook_images/plot_images/offense_bulk.png', scale=2)
fig2.show()

In [17]:
# --- 4c: Speed-Bulk & Offense-Bulk Pearson r by Generation ---
gen_corrs_speed = (
    pokes.groupby('Generation')
    .apply(lambda df: df['Speed'].corr(df['Bulk']))
    .reset_index(name='Pearson_r')
)
gen_corrs_off = (
    pokes.groupby('Generation')
    .apply(lambda df: df['Offense'].corr(df['Bulk']))
    .reset_index(name='Pearson_r')
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=gen_corrs_speed['Generation'], y=gen_corrs_speed['Pearson_r'],
    mode='lines+markers',
    line=dict(color='#5B9BD5', width=4),
    marker=dict(size=14, color='#5B9BD5'),
    name='Speed–Bulk',
))
fig.add_trace(go.Scatter(
    x=gen_corrs_off['Generation'], y=gen_corrs_off['Pearson_r'],
    mode='lines+markers',
    line=dict(color='#E8825C', width=4),
    marker=dict(size=14, color='#E8825C'),
    name='Offense–Bulk',
))
fig.add_hline(y=0, line_dash='dash', line_color='rgba(74,38,112,0.4)')

fig.update_layout(
    title=dict(
        text='<b>Speed–Bulk & Offense–Bulk Pearson r by Generation</b><br>'
             '<sup>More Negative = Stronger Tradeoff</sup>',
        x=0.5, font=dict(size=20)
    ),
    xaxis=dict(title='Generation', tickmode='linear', tick0=1, dtick=1, gridcolor='rgba(0,0,0,0.69)'),
    yaxis=dict(title='Pearson r', gridcolor='rgba(0,0,0,0.69)'),
    template=_T, height=_H, width=_W,
    paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
)

note_speed = '  |  '.join(f"Gen {int(r.Generation)}: {r.Pearson_r:.3f}" for _, r in gen_corrs_speed.iterrows())
note_off   = '  |  '.join(f"Gen {int(r.Generation)}: {r.Pearson_r:.3f}" for _, r in gen_corrs_off.iterrows())
stats_note(fig,
    f'<b>Speed–Bulk r:</b>  {note_speed}<br>'
    f'<b>Offense–Bulk r:</b>  {note_off}',  y_pos=-0.33, bottom_margin=123)

fig.write_image('notebook_images/plot_images/dual_bulk.png', scale=2)
fig.show()

In [17]:
#" --- 4d: Speed Tier × Stat Tier cross-tab heatmap ---
speed_tier_order = ['Slow', 'Medium', 'Fast']
stat_tier_order  = ['Low', 'Mid', 'High', 'Very High']

cross = pd.crosstab(
    pokes['Speed Tier'], pokes['Stat Tier']
).reindex(index=speed_tier_order, columns=stat_tier_order, fill_value=0)

fig = px.imshow(
    cross,
    labels=dict(x='Stat Tier (Overall Power)', y='Speed Tier', color='Count'),
    color_continuous_scale='Blues',
    text_auto=True,
    title='<b>Speed Tier × Stat Tier Cross-Tabulation</b><br>'
          '<sup>Fast Pokémon — are they also high overall power?</sup>',
    template=_T, height=1000, width=1500,
)
fig.update_layout(xaxis=dict(side='bottom'), title_x=0.5)

spd_num  = pokes['Speed Tier'].map({'Slow': 0, 'Medium': 1, 'Fast': 2}).fillna(0)
stat_num = pokes['Stat Tier'].map({'Low': 0, 'Mid': 1, 'High': 2, 'Very High': 3}).fillna(0)
r_tiers  = spd_num.corr(stat_num)

note = (
    f'<b>Total Pokémon in cross-tab: {int(cross.values.sum())}</b>  |  '
    f'Pearson r (Speed Tier vs. Stat Tier): {r_tiers:.3f}<br>'
    f'Diagonal trend = fast Pokémon tend toward higher overall stat tiers (or vice versa)'
)
stats_note(fig, note, y_pos=-0.13, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
)

fig.write_image('notebook_images/plot_images/speed_crosstab.png', scale=2)
fig.show()

In [18]:
# --- 4e: Offense Tier × Stat Tier cross-tab heatmap ---
offense_tier_order = ['Low Offense', 'Mid Offense', 'High Offense']
stat_tier_order    = ['Low', 'Mid', 'High', 'Very High']

off_tier = pd.qcut(pokes['Offensive Total'], q=3, labels=offense_tier_order)

cross = pd.crosstab(
    off_tier, pokes['Stat Tier']
).reindex(index=offense_tier_order, columns=stat_tier_order, fill_value=0)

fig = px.imshow(
    cross,
    labels=dict(x='Stat Tier (Overall Power)', y='Offense Tier', color='Count'),
    color_continuous_scale='Oranges',
    text_auto=True,
    title='<b>Offense Tier × Stat Tier Cross-Tabulation</b><br>'
          '<sup>Do high-offense Pokémon also dominate overall stat tiers?</sup>',
    template=_T, height=1000, width=1500,
)
fig.update_layout(xaxis=dict(side='bottom'), title_x=0.5)

off_num  = off_tier.map({'Low Offense': 0, 'Mid Offense': 1, 'High Offense': 2}).fillna(0)
stat_num = pokes['Stat Tier'].map({'Low': 0, 'Mid': 1, 'High': 2, 'Very High': 3}).fillna(0)
r_tiers  = off_num.corr(stat_num)

note = (
    f'<b>Total Pokémon in cross-tab: {int(cross.values.sum())}</b>  |  '
    f'Pearson r (Offense Tier vs. Stat Tier): {r_tiers:.3f}<br>'
    f'Diagonal trend = high-offense Pokémon tend toward higher overall stat tiers (or vice versa)'
)
stats_note(fig, note, y_pos=-0.13, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
)

fig.write_image('notebook_images/plot_images/offense_crosstab.png', scale=2)
fig.show()

### **3.3.iii.5)** Section 5: Is Power Creep Asymmetric?

---
**Question:** Has offensive power inflated faster than defensive bulk across nine generations?

In [20]:
# --- 5a: Offensive vs. Defensive totals by generation ---
gen_stats = pokes.groupby('Generation').agg(
    Offensive_Mean=('Offensive Total', 'mean'),
    Defensive_Mean=('Defensive Total', 'mean'),
    StatTotal_Mean=('Stat Total', 'mean'),
    OffDef_Ratio=('Offensive/Defensive', 'mean'),
    **{f'{s}_Mean': (s, 'mean') for s in STAT_COLS}
).reset_index()

assert len(gen_stats) == 9, f"Expected 9 generations, got {len(gen_stats)}"

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Mean Offensive vs. Defensive Total by Generation',
                                    'Mean Offensive/Defensive Ratio by Generation'])

fig.add_trace(go.Scatter(
    x=gen_stats['Generation'], y=gen_stats['Offensive_Mean'],
    mode='lines+markers', name='Offensive Total',
    line=dict(color='#F28B82', width=4), marker=dict(size=12),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=gen_stats['Generation'], y=gen_stats['Defensive_Mean'],
    mode='lines+markers', name='Defensive Total',
    line=dict(color='#5B9BD5', width=4), marker=dict(size=12),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=gen_stats['Generation'], y=gen_stats['OffDef_Ratio'],
    mode='lines+markers', name='Off/Def Ratio',
    line=dict(color='#FFD700', width=4), marker=dict(size=12),
), row=1, col=2)
fig.add_hline(y=1.0, line_dash='dash', line_color='rgba(255,255,255,0.4)', row=1, col=2)

fig.update_layout(
    title=dict(text='<b>Is Power Creep Asymmetric?</b>', x=0.5, font=dict(size=20)),
    xaxis=dict(title='Generation', tickmode='linear', tick0=1, dtick=1),
    xaxis2=dict(title='Generation', tickmode='linear', tick0=1, dtick=1),
    yaxis_title='Mean Partial Stat Total',
    yaxis2_title='Ratio  (>1 = more offensive)',
    template=_T, height=777, width=1600,
)

max_off_gen = gen_stats.loc[gen_stats['Offensive_Mean'].idxmax(), 'Generation']
max_def_gen = gen_stats.loc[gen_stats['Defensive_Mean'].idxmax(), 'Generation']
note = (
    f'<b>Peak Offensive Total:</b> Gen {int(max_off_gen)} ({gen_stats["Offensive_Mean"].max():.1f})  |  '
    f'<b>Peak Defensive Total:</b> Gen {int(max_def_gen)} ({gen_stats["Defensive_Mean"].max():.1f})'
)
stats_note(fig, note)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    xaxis2=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis2=dict(gridcolor='rgba(0,0,0,0.69)')
)

fig.write_image('notebook_images/plot_images/stat_inflation_creep.png', scale=2)
fig.show()

In [22]:
# --- 5b: Which individual stats inflated most? ---
stat_colors = {
    'HP': '#91CF60', 'Attack': '#FC8D59', 'Defense': '#5B9BD5',
    'Speed': '#FFD700', 'Special Attack': '#BF80FF', 'Special Defense': '#7FDBFF',
}

fig = make_subplots(rows=2, cols=3,
                    subplot_titles=[f'Mean {s} by Generation' for s in STAT_COLS],
                    vertical_spacing=0.15, horizontal_spacing=0.08)

for i, stat in enumerate(STAT_COLS):
    row, col = divmod(i, 3)
    row += 1; col += 1
    fig.add_trace(go.Scatter(
        x=gen_stats['Generation'], y=gen_stats[f'{stat}_Mean'],
        mode='lines+markers',
        line=dict(color=stat_colors.get(stat, '#888888'), width=4),
        marker=dict(size=12),
        name=stat, showlegend=False,
    ), row=row, col=col)

fig.update_layout(
    title=dict(text='<b>Which Stats Inflated Most? (Power Creep by Stat)</b>', x=0.5, font=dict(size=20)),
    template=_T, height=1000, width=1800,
)
for i in range(1, 7):
    ax = 'xaxis' if i == 1 else f'xaxis{i}'
    fig.update_layout(**{ax: dict(tickmode='linear', tick0=1, dtick=1)})

delta = gen_stats.set_index('Generation')[[f'{s}_Mean' for s in STAT_COLS]].diff().dropna()
fastest = delta.sum().idxmax().replace('_Mean', '')
stats_note(fig, f'<b>Fastest-growing stat (Gen 1→9 cumulative increase):</b>  {fastest}  ({delta.sum().max():.1f} pts)', y_pos=-0.13, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    xaxis2=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis2=dict(gridcolor='rgba(0,0,0,0.69)'),
    xaxis3=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis3=dict(gridcolor='rgba(0,0,0,0.69)'),
     xaxis4=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis4=dict(gridcolor='rgba(0,0,0,0.69)'),
    xaxis5=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis5=dict(gridcolor='rgba(0,0,0,0.69)'),
    xaxis6=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis6=dict(gridcolor='rgba(0,0,0,0.69)'),
)

fig.write_image('notebook_images/plot_images/stat_inflation_all.png', scale=2)
fig.show()

In [28]:
# --- 5c: Stat Total distribution per generation ---
_gen_palette = {str(g): c for g, c in zip(
    sorted(pokes['Generation'].dropna().unique()),
    ['#440154','#46327e','#365c8d','#277f8e','#1fa187','#4ac16d','#a0da39','#fde725','#f5a623']
)}
fig = px.box(
    pokes.assign(Gen=pokes['Generation'].astype(str)),
    x='Gen', y='Stat Total',
    color='Gen', color_discrete_map=_gen_palette,
    title='<b>Stat Total Distribution by Generation</b>',
    labels={'Gen': 'Generation'},
    template=_T, height=777, width=1600,
    points=False,
)
fig.update_layout(showlegend=False, title_x=0.5)

_overall_mean = pokes['Stat Total'].mean()
fig.add_hline(
    y=_overall_mean,
    line_color='#4285f4', line_dash='dash', line_width=2,
    annotation_text=f'All-gen avg: {_overall_mean:.1f}',
    annotation_position='top left',
    annotation_font_color='#000000',
)

note = '  |  '.join(
    f"Gen {int(r.Generation)}: {round(r.StatTotal_Mean, 1)}"
    for _, r in gen_stats.iterrows()
)
stats_note(fig, f'<b>Mean Stat Total per Generation:</b><br>{note}', y_pos=-0.19, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.69)')
)

fig.write_image('notebook_images/plot_images/stat_inflation_totals.png', scale=2)
fig.show()

### **3.3.iii.6)** Section 6: Is the Legendary Gap Closing?

---
**Question:** Have ordinary Pokémon powered up enough that Legendary status no longer signals exceptional strength?

In [65]:
# --- 6a: Stat Total — Legendary vs Non-Legendary ---
assert pokes['Legendary'].any(), "No Legendary=True rows found"

fig = go.Figure()
for leg, label, clr in [(False, 'Non-Legendary', '#5B9BD5'), (True, 'Legendary', '#FFD700')]:
    fig.add_trace(go.Violin(
        y=pokes[pokes['Legendary'] == leg]['Stat Total'],
        name=label, fillcolor=clr,
        line=dict(color='white', width=3),
        opacity=0.75, box_visible=True, meanline_visible=True, points=False,
    ))

fig.update_layout(
    title=dict(text='<b>Stat Total: Legendary vs. Non-Legendary</b>', x=0.5, font=dict(size=20)),
    yaxis_title='Stat Total',
    template=_T, height=777, width=1800,
    violinmode='overlay', showlegend=True,
)

summary  = pokes.groupby('Legendary')['Stat Total'].agg(['mean', 'median', 'min', 'max']).round(1)
leg_row  = summary.loc[True]
nleg_row = summary.loc[False]
gap = leg_row['mean'] - nleg_row['mean']
note = (
    f"<b>Non-Legendary:</b> mean={nleg_row['mean']}, median={nleg_row['median']}, "
    f"range=[{nleg_row['min']}, {nleg_row['max']}]<br>"
    f"<b>Legendary:</b> mean={leg_row['mean']}, median={leg_row['median']}, "
    f"range=[{leg_row['min']}, {leg_row['max']}]<br>"
    f"<b>Mean gap (Legendary − Non-Legendary):</b> {gap:.1f}"
)
stats_note(fig, note, y_pos=-0.13, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
)

fig.write_image('notebook_images/plot_images/legendary_stattotals.png', scale=2)
fig.show()

In [68]:
# --- 6b: Legendary gap by generation ---
gen_leg = (
    pokes.groupby(['Generation', 'Legendary'])['Stat Total']
    .mean().unstack('Legendary')
    .rename(columns={False: 'Non-Legendary', True: 'Legendary'})
)
gen_leg['Gap'] = gen_leg['Legendary'] - gen_leg['Non-Legendary']
gen_leg = gen_leg.reset_index()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Mean Stat Total by Generation × Legendary Status',
                                    'Legendary Stat Gap by Generation\n(Legendary − Non-Legendary)'])

for col_key, label, clr in [('Non-Legendary', 'Non-Legendary', '#5B9BD5'),
                              ('Legendary', 'Legendary', '#FFD700')]:
    fig.add_trace(go.Scatter(
        x=gen_leg['Generation'], y=gen_leg[col_key],
        mode='lines+markers', name=label,
        line=dict(color=clr, width=2), marker=dict(size=8),
    ), row=1, col=1)

fig.add_trace(go.Bar(
    x=gen_leg['Generation'], y=gen_leg['Gap'],
    name='Gap', marker_color='#FF8C00',
), row=1, col=2)
fig.add_hline(y=0, line_dash='dash', line_color='rgba(255,255,255,0.4)', row=1, col=2)

fig.update_layout(
    title=dict(text='<b>Is the Legendary Gap Closing?</b>', x=0.5, font=dict(size=20)),
    xaxis=dict(title='Generation', tickmode='linear', tick0=1, dtick=1),
    xaxis2=dict(title='Generation', tickmode='linear', tick0=1, dtick=1),
    yaxis_title='Mean Stat Total',
    yaxis2_title='Stat Total Gap',
    template=_T, height=777, width=1800,
)

gaps = gen_leg.set_index('Generation')['Gap']
note = '  |  '.join(f"Gen {int(g)}: {v:.1f}" for g, v in gaps.items())
stats_note(fig, f'<b>Legendary gap per generation:</b><br>{note}', y_pos=-0.19, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
     xaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    xaxis2=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis2=dict(gridcolor='rgba(0,0,0,0.69)')
)

fig.write_image('notebook_images/plot_images/legendary_gap.png', scale=2)
fig.show()

In [104]:
# --- 6c: Top 25 Legendary vs Top 25 Non-Legendary by Stat Total ---
_top25_leg    = pokes[pokes['Legendary']].nlargest(25, 'Stat Total').assign(Group='Legendary')
_top25_nonleg = pokes[~pokes['Legendary']].nlargest(25, 'Stat Total').assign(Group='Non-Legendary')
top50_combined = pd.concat([_top25_leg, _top25_nonleg])

_group_colors = {'Legendary': '#6F35FC', 'Non-Legendary': '#F7D02C'}

fig = px.strip(
    top50_combined,
    x='Generation', y='Stat Total',
    color='Group',
    color_discrete_map=_group_colors,
    hover_name='Pokemon',
    hover_data=['Role', 'Is Variation', 'Stat Tier'],
    title='<b>Top 25 Legendary vs. Top 25 Non-Legendary Pokémon by Stat Total</b>',
    template=_T, height=_H, width=_W,
)
fig.update_layout(xaxis=dict(tickmode='linear', tick0=1, dtick=1), title_x=0.5)
fig.update_traces(marker=dict(size=17))

# Build aligned two-row note — prefix padded to same width so gen columns line up
_all_gens      = sorted(pokes['Generation'].dropna().unique().astype(int))
_leg_counts    = _top25_leg['Generation'].value_counts()
_nonleg_counts = _top25_nonleg['Generation'].value_counts()

def _gen_row(counts):
    return "  |  ".join(f"Gen {g}: {str(counts.get(g, 0)).rjust(2)}" for g in _all_gens)

# "Legendary:    " == "Non-Legendary:" == 14 chars → add 1 space each → 15-char prefix
note = (
    f"Legendary:     {_gen_row(_leg_counts)}<br>"
    f"Non-Legendary: {_gen_row(_nonleg_counts)}"
)
stats_note(fig, note, y_pos=-0.17, bottom_margin=123)
fig.layout.annotations[-1].update(font=dict(family='Courier New', size=12))

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='black'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.69)'),
)

fig.write_image('notebook_images/plot_images/legendary_tops.png', scale=2)
fig.show()

### **3.3.iii.7)** Section 7: ML Capstone — Unsupervised Clustering

---
**Question:** Without using any labels, can the data's own patterns rediscover the role archetypes we defined — and reveal unexpected groupings?

**Method:** Gaussian Mixture Model (GMM) clustering on an expanded feature set (6 stat z-scores + scaled BMI + scaled Physical/Special ratio + scaled Offensive/Defensive ratio + one-hot encoded Type 1), with the optimal number of components selected automatically via the Bayesian Information Criterion (BIC). Visualized via PCA dimensionality reduction.

In [4]:
# --- 7a: Feature engineering + GMM with BIC ---
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

EXTRA_COLS = ['BMI', 'Physical/Special', 'Offensive/Defensive']

cluster_df = pokes.dropna(subset=ZSCORE_COLS + EXTRA_COLS + ['Type 1']).copy()

# Scale continuous extras
scaler = StandardScaler()
scaled_extras = scaler.fit_transform(cluster_df[EXTRA_COLS])
for i, col in enumerate(EXTRA_COLS):
    cluster_df[f'{col}_scaled'] = scaled_extras[:, i]

# One-hot encode Type 1
type1_dummies = pd.get_dummies(cluster_df['Type 1'], prefix='T1').astype(float)
cluster_df = pd.concat([cluster_df, type1_dummies], axis=1)

FEATURE_COLS = (ZSCORE_COLS
                + [f'{c}_scaled' for c in EXTRA_COLS]
                + [c for c in type1_dummies.columns])
X = cluster_df[FEATURE_COLS].values

print(f"Feature matrix: {X.shape[0]} rows × {X.shape[1]} features")

# BIC search across K=2..15
K_range = range(2, 16)
bics = []
for k in K_range:
    gmm = GaussianMixture(n_components=k, covariance_type='diag',
                           random_state=42, n_init=5, max_iter=300)
    gmm.fit(X)
    bics.append(gmm.bic(X))

best_k = list(K_range)[bics.index(min(bics))]

fig = go.Figure(go.Scatter(
    x=list(K_range), y=bics, mode='lines+markers',
    line=dict(color='#5B9BD5', width=7), marker=dict(size=13),
))
fig.add_vline(x=best_k, line_dash='dash', line_color='#F28B82', line_width=2,
              annotation_text=f'Best K={best_k}', annotation_position='top right',
              annotation_font_color='#F28B82')
fig.update_layout(
    title=dict(text='<b>Choosing Optimal K — BIC Curve</b>', x=0.5),
    xaxis=dict(title='Number of Components (K)', tickmode='linear', tick0=2, dtick=1),
    yaxis_title='BIC (lower is better)',
    template=_T, height=_H, width=_W,
)

note = (
    f'<b>Best K by BIC:</b> {best_k}  (BIC: {min(bics):,.0f})<br>'
    f'<b>Features ({len(FEATURE_COLS)}):</b> 6 stat z-scores + 3 scaled ratios (BMI, Phys/Spec, Off/Def) '
    f'+ {len(type1_dummies.columns)} one-hot Type 1 columns<br>'
    f'<b>Covariance type:</b> diagonal  |  <b>K range tested:</b> 2–15'
)
stats_note(fig, note, y_pos=-0.2, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
)

fig.write_image('notebook_images/plot_images/clustering_k.png', scale=2)
fig.show()

print(f"\nBIC selected K={best_k}")

Feature matrix: 1231 rows × 27 features



BIC selected K=14


In [5]:
# --- 7b: Fit final GMM + PCA projection ---
gmm_final = GaussianMixture(n_components=best_k, covariance_type='diag',
                             random_state=42, n_init=5, max_iter=300)
cluster_df['Cluster'] = gmm_final.fit_predict(X)
cluster_df['Cluster_Prob'] = gmm_final.predict_proba(X).max(axis=1)

pca = PCA(n_components=2, random_state=42)
components = pca.fit_transform(X)
cluster_df['PC1'] = components[:, 0]
cluster_df['PC2'] = components[:, 1]

n_clusters = best_k
cluster_sizes = cluster_df['Cluster'].value_counts().sort_index()

# Sample 13 per cluster, or all members if cluster has fewer than 13
_target_n = 13
sample_df = (cluster_df.groupby('Cluster', group_keys=False)
             .apply(lambda g: g.sample(n=min(_target_n, len(g)), random_state=42)))

_small_clusters = {c: int(n) for c, n in cluster_sizes.items() if n < _target_n}
if _small_clusters:
    _small_note = '  |  '.join(f'Cluster {c}: {n}' for c, n in sorted(_small_clusters.items()))
    _subtitle = (f'<sup>Random sample of {_target_n} Pokémon per cluster '
                 f'(showing all: {_small_note})</sup>')
else:
    _subtitle = f'<sup>Random sample of {_target_n} Pokémon per cluster</sup>'

_palette = {str(c): ROLE_COLORS[c % len(ROLE_COLORS)]
            for c in range(n_clusters)}

fig = px.scatter(
    sample_df,
    x='PC1', y='PC2',
    color=sample_df['Cluster'].astype(str),
    color_discrete_map=_palette,
    hover_name='Pokemon',
    hover_data=['Role', 'Type 1', 'Legendary', 'Stat Total', 'Generation', 'BMI', 'Cluster_Prob'],
    title=f'<b>GMM Clusters (K={best_k}, BIC-selected) — PCA Projection</b><br>{_subtitle}',
    template=_T, opacity=0.8,
    height=1000, width=2000,
    category_orders={'color': [str(c) for c in range(n_clusters)]},
)
fig.update_layout(legend_title='Cluster', title_x=0.5)
fig.update_traces(marker=dict(size=20))

# Draw 2σ confidence ellipses from full cluster data
_theta = np.linspace(0, 2 * np.pi, 100)
for c in range(n_clusters):
    _mask = cluster_df['Cluster'] == c
    _pts = cluster_df.loc[_mask, ['PC1', 'PC2']].values
    if len(_pts) < 3:
        continue
    _mean = _pts.mean(axis=0)
    _cov = np.cov(_pts.T)
    _eigvals, _eigvecs = np.linalg.eigh(_cov)
    _order = _eigvals.argsort()[::-1]
    _eigvals = _eigvals[_order]
    _eigvecs = _eigvecs[:, _order]
    _angle = np.arctan2(_eigvecs[1, 0], _eigvecs[0, 0])
    # 2σ ellipse (covers ~86% of 2D Gaussian)
    _a = 2 * np.sqrt(_eigvals[0])
    _b = 2 * np.sqrt(_eigvals[1])
    _ellipse_x = _a * np.cos(_theta)
    _ellipse_y = _b * np.sin(_theta)
    _rot = np.array([[np.cos(_angle), -np.sin(_angle)],
                     [np.sin(_angle),  np.cos(_angle)]])
    _rotated = _rot @ np.array([_ellipse_x, _ellipse_y])
    _color = _palette[str(c)]
    fig.add_trace(go.Scatter(
        x=_rotated[0] + _mean[0], y=_rotated[1] + _mean[1],
        mode='lines', line=dict(color=_color, width=2, dash='dot'),
        showlegend=False, hoverinfo='skip',
    ))
    fig.add_annotation(
        x=_mean[0], y=_mean[1] + _b * 0.6,
        text=f'<b>C{c}</b>', showarrow=False,
        font=dict(color=_color, size=14),
    )

sizes_str = '  |  '.join(f"Cluster {c}: {n}" for c, n in cluster_sizes.items())
pca_var = sum(pca.explained_variance_ratio_[:2])
mean_prob = cluster_df['Cluster_Prob'].mean()
note = (
    f'<b>Cluster sizes (full):</b>  {sizes_str}<br>'
    f'<b>PCA explained variance:</b>  '
    f'PC1={pca.explained_variance_ratio_[0]:.1%}  |  '
    f'PC2={pca.explained_variance_ratio_[1]:.1%}  |  '
    f'Total={pca_var:.1%}<br>'
    f'<b>Mean assignment confidence:</b> {mean_prob:.1%}  |  '
    f'<b>Showing:</b> {len(sample_df)} of {len(cluster_df)} total'
)
stats_note(fig, note, y_pos=-0.15, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
)

fig.write_image('notebook_images/plot_images/clustering_clustersamples.png', scale=2)
fig.show()

In [6]:
# --- 7c: How well do data-driven clusters match rule-based roles? ---
cross = pd.crosstab(cluster_df['Cluster'], cluster_df['Role'])[ROLE_ORDER]

fig = px.imshow(
    cross,
    labels=dict(x='Role (rule-based label)',
                y=f'Cluster (GMM, K={best_k})',
                color='Count'),
    color_continuous_scale='Blues',
    text_auto=True,
    title=f'<b>GMM Cluster vs. Role Label (K={best_k})</b>',
    template=_T, height=_H, width=_W,
)
fig.update_layout(xaxis=dict(side='bottom'), title_x=0.5)

dominant_roles = cross.idxmax(axis=1)
_items = [f"C{c}: {r}" for c, r in dominant_roles.items()]
_mid = (len(_items) + 1) // 2
_row1 = '  |  '.join(_items[:_mid])
_row2 = '  |  '.join(_items[_mid:])
note = f'<b>Dominant role per cluster:</b><br>{_row1}<br>{_row2}'
stats_note(fig, note, y_pos=-0.19, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
)

fig.write_image('notebook_images/plot_images/clustering_heatmap.png', scale=2)
fig.show()

In [94]:
 # --- 7d: What does each cluster "look like" statistically? ---
_display_cols = STAT_COLS + EXTRA_COLS

cluster_melted = (
    cluster_df.groupby('Cluster')[_display_cols].mean()
    .reset_index()
    .melt(id_vars='Cluster', var_name='Feature', value_name='Mean Value')
)
cluster_melted['Cluster'] = cluster_melted['Cluster'].astype(str)

_bar_colors = {str(c): ROLE_COLORS[int(c) % len(ROLE_COLORS)]
               for c in cluster_melted['Cluster'].unique()}

fig = px.bar(
    cluster_melted, x='Feature', y='Mean Value',
    color='Cluster', barmode='group',
    color_discrete_map=_bar_colors,
    title=f'<b>Mean Feature Values per Cluster (GMM, K={best_k})</b>',
    template=_T, height=1000, width=2000,
    log_y=True,
)
fig.update_layout(xaxis_title='Feature', yaxis_title='Mean Value (log scale)', title_x=0.5)

# Top type per cluster
type_dom = cluster_df.groupby('Cluster')['Type 1'].agg(lambda x: x.value_counts().index[0])
note = ('<b>Dominant Type 1 per cluster:</b>  '
        + '  |  '.join(f"C{c}: {t}" for c, t in type_dom.items()))

stats_note(fig, note, y_pos=-0.13, bottom_margin=123)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
)

fig.write_image('notebook_images/plot_images/clustering_featurevalues.png', scale=2)
fig.show()

In [102]:
# --- 7e: Cluster identity definitions ---
# Curated definitions based on stat profiles, type composition, and domain knowledge

_stat_means  = cluster_df.groupby('Cluster')[STAT_COLS].mean()
_extra_means = cluster_df.groupby('Cluster')[EXTRA_COLS].mean()
_all_means   = pd.concat([_stat_means, _extra_means], axis=1)
_dom_type    = cluster_df.groupby('Cluster')['Type 1'].agg(lambda x: x.value_counts().index[0])
_leg_pct     = (cluster_df.groupby('Cluster')['Legendary'].mean() * 100).round(0).astype(int)
_bst_mean    = cluster_df.groupby('Cluster')['Stat Total'].mean().round(0).astype(int)

# --- Curated cluster identities ---
CLUSTER_NAMES = {
    0:  'Fairy Fortress Elite',
    1:  'Dragon Powerhouses',
    2:  'Extreme Wall Outlier',
    3:  'Ghost Generalists',
    4:  'Electric Speedsters',
    5:  'Blue-Collar Brawlers',
    6:  'Armored Heavyweights',
    7:  'Low-Stat Midtiers',
    8:  'Pixie Wardens',
    9:  'Rooted Slowpokes',
    10: 'Toxic Heavies',
    11: 'Shadow Strikers',
    12: 'Psychic Cannons',
    13: 'Fragile Normals',
}

CLUSTER_DEFS = {
    0:  'Rare, high-BST Fairy specialists with towering Sp.Def (104 avg). '
        'Small elite cluster separated from main Fairy group by extreme special bulk.',
    1:  'All-Dragon cluster — highest BST of any group (545). Elevated across every stat, '
        'especially Attack (107). Nearly 1-in-3 are Legendary.',
    2:  'Single Legendary outlier (Cosmoem). Def and Sp.Def both 131, but near-zero offenses. '
        'A statistical extreme the model correctly isolated.',
    3:  'All-Ghost cluster with no dominant stat — versatile role coverage spanning '
        'Tanks, Balanced, and Sp. Sweepers. Unified by type, not stat shape.',
    4:  'All-Electric cluster built to outspeed and blast. Speed (88) and Sp.Atk (89) lead; '
        'the most offensively skewed group (Off/Def = 1.29).',
    5:  'Water (73%) + Fighting (27%) physical generalists — the largest cluster (n=205). '
        'Reliable middleweight stats with a physical lean (Phys/Spec = 1.2).',
    6:  'Rock/Ground/Steel/Ice defensive wall — heaviest cluster (BMI 69), highest Def (95), '
        'slowest Speed (62). The classic slow physical bulk archetype.',
    7:  'Bug (54%) + Fire (46%) united by low BST (425). The "average Pokémon" cluster — '
        'no extreme stats, mostly unevolved or single-stage species.',
    8:  'Main Fairy archetype — specially defensive (Sp.Def 90) but lighter build (BMI 21) '
        'and lower BST (462) than C0\'s elite Fairy tanks.',
    9:  'All-Grass cluster defined by poor Speed (61 avg). Slow, balanced generalists — '
        'the type and stat archetype are inseparable.',
    10: 'Poison (83%) + Normal (17%) with no standout stats but surprising bulk (BMI 49). '
        'A residual cluster of types without a strong stat fingerprint.',
    11: 'All-Dark cluster with above-average Attack (88) and Speed (79). '
        'Offensively skewed (Off/Def = 1.17) with evenly split roles.',
    12: 'All-Psychic special warfare experts — highest Sp.Atk (101) and 2nd-highest BST (486). '
        '35% Legendary. Specially oriented (Phys/Spec = 0.79).',
    13: 'Normal (92%) + Flying — lowest BST of any cluster (412). Below average in Def, '
        'Sp.Atk, and Sp.Def. The "early-route everymon" archetype.',
}

# --- Build table matching Combat Role Definitions style ---
header_color = '#735797'
row_colors   = ['#B7B7CE', '#9696a8']
_cluster_colors = [ROLE_COLORS[c % len(ROLE_COLORS)] for c in sorted(CLUSTER_NAMES.keys())]

fig = go.Figure(go.Table(
    columnwidth=[140, 180, 70, 60, 60, 700],
    header=dict(
        values=['<b>Cluster</b>', '<b>Name</b>', '<b>Size</b>',
                '<b>BST</b>', '<b>Leg%</b>', '<b>Definition</b>'],
        fill_color=header_color,
        font=dict(color='white', size=24),
        align='center',
        height=36,
    ),
    cells=dict(
        values=[
            [f'<b>C{c}</b>' for c in sorted(CLUSTER_NAMES.keys())],
            [f'<b>{CLUSTER_NAMES[c]}</b>' for c in sorted(CLUSTER_NAMES.keys())],
            [f'{cluster_sizes[c]}' for c in sorted(CLUSTER_NAMES.keys())],
            [f'{_bst_mean[c]}' for c in sorted(CLUSTER_NAMES.keys())],
            [f'{_leg_pct[c]}%' for c in sorted(CLUSTER_NAMES.keys())],
            [CLUSTER_DEFS[c] for c in sorted(CLUSTER_NAMES.keys())],
        ],
        fill_color=[
            _cluster_colors, _cluster_colors, _cluster_colors,
            _cluster_colors, _cluster_colors,
            [row_colors[i % 2] for i in range(len(CLUSTER_NAMES))],
        ],
        font=dict(color='white', size=15),
        align=['center', 'left', 'center', 'center', 'center', 'left'],
        height=44,
    ),
))
fig.update_layout(
    title=dict(text=f'<b>Cluster Identity Definitions (GMM, K={best_k})</b>', x=0.5),
    template=_T,
    height=max(700, 130 + len(CLUSTER_NAMES) * 60),
    width=_W,
    margin=dict(l=20, r=20, t=60, b=0),
)
fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(0,0,0,0.9)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.9)'),
)

fig.write_image('notebook_images/plot_images/clustering_definitions.png', scale=2)
fig.show()